# Multilingual Document OCR + MRZ Parser
### Domino Data Lab · Arabic / French / English · Fully Offline · v5

---

## All bugs fixed in this version

| Version | Error | Cause | Fix |
|---------|-------|-------|-----|
| v1 | `AssertionError: param lang must in dict_keys, but got ar,fr,en` | PaddleOCR rejects comma-separated lang | Two engines: `lang='ar'` + `lang='fr'` |
| v2–v3 | `ConnectionError` even after passing `det_model_dir` | PaddleOCR ignored the parameter | Copied models into `~/.paddleocr/whl/` cache |
| v4 | Still `ConnectionError`, models went to wrong folder | `Path.home()` returns wrong path inside Domino conda env | **v5: import `BASE_DIR` from PaddleOCR's own source + monkey-patch downloader** |

## How v5 works
1. **`BASE_DIR` import** — we read the cache path directly from PaddleOCR source. No guessing.
2. **Auto-copy** — if models are not in cache, we copy them from `/mnt/data/models/` automatically.
3. **Download blocker** — we patch `requests.Session.send` before creating any engine.  
   If PaddleOCR ever tries to reach the internet, you see a clear diagnostic error instead of `ConnectionError`.

## Before running
Complete Steps 1–3 in **INSTRUCTIONS.txt**:
1. **Laptop terminal** — download + extract the 3 model `.tar` files
2. **Domino browser** — upload model folders to `/mnt/data/models/` and images to `/mnt/data/documents/`
3. **Domino terminal** — `pip install` the packages

---
## Cell 1 — Imports

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import json
import logging
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Cell 1 complete — imports OK')

---
## Cell 2 — Find PaddleOCR Cache, Copy Models, Block Downloads

This cell does three things automatically:

**1. Finds the exact cache path** by importing `BASE_DIR` from PaddleOCR's own source.  
Previous versions guessed with `Path.home()` — wrong inside Domino's conda environment.

**2. Copies model files** from `/mnt/data/models/` into the correct cache sub-folders  
if they are not already there.

**3. Blocks all downloads** by patching `requests.Session.send`.  
If PaddleOCR tries to reach the internet you will see a clear `❌` error instead of  
a confusing `ConnectionError` buried in a long traceback.

In [ ]:
import requests as _requests

# =============================================================================
# A) FIND BASE_DIR — the exact path PaddleOCR uses for its model cache
# =============================================================================

# Import BASE_DIR from PaddleOCR's own source — no guessing, no Path.home()
# This is the ONLY reliable way to find the correct path across all environments.
try:
    from paddleocr.paddleocr import BASE_DIR as _PADDLE_BASE
    PADDLE_BASE = Path(_PADDLE_BASE)
    print(f'✓ PaddleOCR BASE_DIR (from source): {PADDLE_BASE}')
except Exception as e:
    # Fallback: try to find it from the paddleocr package location itself
    import paddleocr as _poc
    PADDLE_BASE = Path(_poc.__file__).parent.parent.parent.parent / '.paddleocr'
    print(f'⚠ BASE_DIR import failed ({e}), using fallback: {PADDLE_BASE}')

# =============================================================================
# B) DEFINE CACHE PATHS — exact sub-folder structure PaddleOCR expects
#    PaddleOCR checks these folders BEFORE attempting any download.
#    If inference.pdmodel is present here, NO network call is made.
# =============================================================================

CACHE_DET   = PADDLE_BASE / 'whl' / 'det' / 'multilingual' / 'Multilingual_PP-OCRv3_det_infer'
CACHE_AR    = PADDLE_BASE / 'whl' / 'rec' / 'arabic'        / 'arabic_PP-OCRv3_rec_infer'
CACHE_LATIN = PADDLE_BASE / 'whl' / 'rec' / 'latin'         / 'latin_PP-OCRv3_rec_infer'

print(f'  Cache det   : {CACHE_DET}')
print(f'  Cache ar    : {CACHE_AR}')
print(f'  Cache latin : {CACHE_LATIN}')

# =============================================================================
# C) SOURCE PATHS — where you uploaded model folders in Step 2
# =============================================================================

UPLOADED    = Path('/mnt/data/models')                              # edit if different
SRC_DET     = UPLOADED / 'Multilingual_PP-OCRv3_det_infer'
SRC_AR      = UPLOADED / 'arabic_PP-OCRv3_rec_infer'
SRC_LATIN   = UPLOADED / 'latin_PP-OCRv3_rec_infer'

# =============================================================================
# D) AUTO-COPY — copy models into cache if not already there
# =============================================================================

def _has_model_files(folder: Path) -> bool:
    """Return True if the folder contains inference.pdmodel."""
    return folder.exists() and (folder / 'inference.pdmodel').exists()


def ensure_cached(src: Path, dst: Path, label: str) -> None:
    """
    Copy model folder from uploaded location into the PaddleOCR cache.
    Skipped if inference.pdmodel already exists at the destination.
    """
    if _has_model_files(dst):
        print(f'  ✓ {label:35s} already in cache')
        return

    if not src.exists():
        raise FileNotFoundError(
            f'\n\n❌ Model source not found: {src}\n'
            f'   Upload the model folder to /mnt/data/models/ first (Step 2 in INSTRUCTIONS.txt).\n'
        )

    if not (src / 'inference.pdmodel').exists():
        raise FileNotFoundError(
            f'\n\n❌ inference.pdmodel missing inside: {src}\n'
            f'   The folder exists but is empty or incomplete.\n'
            f'   Re-extract the .tar file and re-upload all three files:\n'
            f'     inference.pdmodel\n'
            f'     inference.pdiparams\n'
            f'     inference.pdiparams.info\n'
        )

    print(f'  → Copying {label} to cache …')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(str(dst))
    shutil.copytree(str(src), str(dst))

    # Verify copy succeeded
    if not _has_model_files(dst):
        raise RuntimeError(f'Copy appeared to succeed but files not found at {dst}')
    print(f'  ✓ {label:35s} copied and verified')


print('\nCopying models to PaddleOCR cache …')
ensure_cached(SRC_DET,   CACHE_DET,   'Detection model')
ensure_cached(SRC_AR,    CACHE_AR,    'Arabic rec model')
ensure_cached(SRC_LATIN, CACHE_LATIN, 'Latin rec model (fr/en)')

# =============================================================================
# E) MONKEY-PATCH: block ALL outbound downloads as a safety net
#    If PaddleOCR still tries to download, you get a clear diagnostic error
#    instead of a buried ConnectionError.
# =============================================================================

_orig_send = _requests.Session.send

def _block_paddle_downloads(self, request, **kwargs):
    url = getattr(request, 'url', str(request))
    if 'bcebos.com' in url or ('paddleocr' in url and 'http' in url):
        raise RuntimeError(
            f'\n\n❌ PaddleOCR tried to download from the internet!\n'
            f'   URL attempted : {url}\n\n'
            f'   This means a model file is MISSING from the cache.\n'
            f'   Expected cache root: {PADDLE_BASE}\n\n'
            f'   Check which model is missing:\n'
            f'     Detection  : {CACHE_DET}\n'
            f'     Arabic rec : {CACHE_AR}\n'
            f'     Latin rec  : {CACHE_LATIN}\n\n'
            f'   Fix: re-upload missing model to /mnt/data/models/ and re-run Cell 2.\n'
        )
    return _orig_send(self, request, **kwargs)

_requests.Session.send = _block_paddle_downloads

# =============================================================================
# F) PATHS AND CONSTANTS FOR REST OF NOTEBOOK
# =============================================================================

INPUT_DIR    = Path('/mnt/data/documents')      # where your images are
OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

MAX_DIMENSION    = 1280
CPU_THREADS      = os.cpu_count() or 4
SUPPORTED_EXT    = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

print('\n✓ Download blocker active — any download attempt will show a clear ❌ error')
print(f'\n✓ Cell 2 complete')
print(f'  Input dir    : {INPUT_DIR}')
print(f'  CPU threads  : {CPU_THREADS}')

---
## Cell 3 — Image Preprocessing

In [ ]:
def load_and_resize(
    image_path: str | Path,
    max_dim: int = MAX_DIMENSION,
) -> np.ndarray:
    """
    Load an image and downscale if the longest side exceeds max_dim.
    cv2.INTER_AREA gives best quality when shrinking.
    Resizing to 1280px roughly halves CPU inference time on large scans.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')
    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)
    if scale < 1.0:
        img = cv2.resize(
            img,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    return img


print('✓ Cell 3 complete — load_and_resize defined')

---
## Cell 4 — Build the Two OCR Engines

**Why two engines?**  
PaddleOCR only accepts **one language code per instance**.  
`lang='ar,fr,en'` raises `AssertionError`. We use:

| Engine | `lang` | Recognises | Cache path |
|--------|--------|------------|-----------|
| `ocr_ar` | `'ar'` | Arabic script | `rec/arabic/arabic_PP-OCRv3_rec_infer` |
| `ocr_latin` | `'fr'` | French + English | `rec/latin/latin_PP-OCRv3_rec_infer` |

**Why no `det_model_dir` / `rec_model_dir` parameters?**  
Those parameters were being silently ignored in previous versions.  
In v5 the models are in the exact cache PaddleOCR checks first, so they are found automatically.  
The download blocker (Cell 2) ensures no network call is ever made.

> ⏱ This cell takes 10–30 seconds. Watch for `✓` lines — NO `Downloading` messages.

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(lang: str) -> PaddleOCR:
    """
    Create a single-language PaddleOCR engine.

    Models are read from the cache populated in Cell 2.
    The download blocker (also Cell 2) ensures no network call is made.

    Parameters
    ----------
    lang : 'ar' for Arabic, 'fr' for French + English

    Returns
    -------
    PaddleOCR instance (offline)
    """
    return PaddleOCR(
        lang=lang,
        use_angle_cls=False,    # documents assumed upright — disables cls model download
        use_gpu=False,          # CPU-only
        enable_hpi=True,        # ONNX Runtime High-Performance Inference
        cpu_threads=CPU_THREADS,
        show_log=False,
    )


print('Initialising Arabic engine  (lang="ar") …')
print('If you see a ❌ error here, re-run Cell 2 to diagnose missing model files.')
ocr_ar = _make_engine('ar')
print('✓ Arabic engine ready')

print()
print('Initialising Latin engine   (lang="fr") …')
ocr_latin = _make_engine('fr')
print('✓ Latin engine ready  (French + English)')

print()
print('✓ Cell 4 complete — both engines ready, fully offline')

---
## Cell 5 — MRZ Detection and Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Scan OCR lines for ICAO MRZ candidates.

    MRZ lines contain only A-Z, 0-9, '<' and are at least 30 chars.
    (TD1=30, TD2=36, passport TD3=44)
    Spaces are removed first because OCR sometimes inserts them.
    """
    mrz: list[str] = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz.append(cleaned)
    return mrz


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Parse and validate MRZ lines using OmniMRZ (ICAO 9303).

    Returns
    -------
    dict with: raw_mrz, parsed_fields, validation, (optional) error
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — MRZ will not be parsed structurally.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    try:
        mrz_obj = MRZ('\n'.join(mrz_lines))

        fields: dict[str, Any] = {
            'document_type':    getattr(mrz_obj, 'document_type',    None),
            'issuing_country':  getattr(mrz_obj, 'issuing_country',  None),
            'document_number':  getattr(mrz_obj, 'document_number',  None),
            'surname':          getattr(mrz_obj, 'surname',          None),
            'given_names':      getattr(mrz_obj, 'given_names',      None),
            'nationality':      getattr(mrz_obj, 'nationality',      None),
            'birth_date':       getattr(mrz_obj, 'birth_date',       None),
            'sex':              getattr(mrz_obj, 'sex',              None),
            'expiry_date':      getattr(mrz_obj, 'expiry_date',      None),
            'optional_data_1':  getattr(mrz_obj, 'optional_data_1', None),
            'optional_data_2':  getattr(mrz_obj, 'optional_data_2', None),
        }

        validation: dict[str, Any] = {}
        for attr in dir(mrz_obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(mrz_obj, attr)
                except Exception:
                    pass

        overall = getattr(mrz_obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall

        return {'raw_mrz': mrz_lines, 'parsed_fields': fields, 'validation': validation}

    except Exception as exc:
        log.warning('OmniMRZ parsing failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ Cell 5 complete — MRZ helpers defined')

---
## Cell 6 — Single-Document Processor (dual engine)

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """
    Run one PaddleOCR engine and return flat list of text strings.
    PaddleOCR output: [ [ [box, (text, score)], ... ] ]
    """
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = item[1][0] if isinstance(item[1], (list, tuple)) else str(item[1])
                lines.append(txt)
    return lines


def process_document(
    image_path:   str | Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
) -> dict[str, Any]:
    """
    Run OCR with both engines, merge results, detect and parse MRZ.

    Steps:
      1. Load + resize image.
      2. engine_ar   → Arabic text lines.
      3. engine_latin → French/English text lines.
      4. Merge lists (Arabic first), remove exact duplicates.
      5. Scan merged lines for MRZ pattern.
      6. If >= 2 MRZ lines: parse with OmniMRZ.

    Returns
    -------
    dict: file_name, full_text, has_mrz, mrz_data, error, elapsed_sec
    """
    image_path = Path(image_path)
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()
    try:
        img         = load_and_resize(image_path, max_dim)
        lines_ar    = _extract_lines(engine_ar,    img)
        lines_latin = _extract_lines(engine_latin, img)

        seen: set[str]        = set()
        text_lines: list[str] = []
        for line in lines_ar + lines_latin:
            if line not in seen:
                seen.add(line)
                text_lines.append(line)

        result['full_text'] = '\n'.join(text_lines)

        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  — No MRZ found')

    except Exception as exc:
        log.error('Error processing %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ Cell 6 complete — process_document defined')

---
## Cell 7 — Test with One Sample Image
Change `SAMPLE_IMAGE` to one of your files. Quick check before running the full batch.

In [ ]:
SAMPLE_IMAGE = Path('/mnt/data/documents/sample.jpg')   # ← change me

if SAMPLE_IMAGE.exists():
    print(f'Testing: {SAMPLE_IMAGE.name}')
    r = process_document(SAMPLE_IMAGE, ocr_ar, ocr_latin)

    print(f'\n  File       : {r["file_name"]}')
    print(f'  Has MRZ    : {r["has_mrz"]}')
    print(f'  Time       : {r["elapsed_sec"]}s')
    print(f'  Error      : {r["error"]}')
    print(f'\n  --- OCR TEXT ---')
    print(r['full_text'] or '(nothing detected)')

    if r['has_mrz']:
        print(f'\n  --- MRZ ---')
        print(json.dumps(r['mrz_data'], ensure_ascii=False, indent=2, default=str))

    print('\n✓ Cell 7 complete')
else:
    print(f'[SKIP] {SAMPLE_IMAGE} not found.')
    print('Change SAMPLE_IMAGE above to a real path, or go to Cell 8.')

---
## Cell 8 — Batch Processing (all images in INPUT_DIR)

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """Return sorted list of images from a file or directory (recursive)."""
    if input_path.is_file():
        return [input_path]
    images = sorted(
        p for p in input_path.rglob('*') if p.suffix.lower() in SUPPORTED_EXT
    )
    if not images:
        print(f'WARNING: no images found in {input_path}')
    return images


images = collect_images(INPUT_DIR)
print(f'Found {len(images)} image(s) in {INPUT_DIR}:')
for p in images:
    print(f'  {p.name}')

print()
all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  %.2fs | has_mrz=%s | error=%s', r['elapsed_sec'], r['has_mrz'], r['error'])

total = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Cell 8 complete — {len(all_results)} doc(s) in {total:.1f}s')

---
## Cell 9 — Save JSON

In [ ]:
def save_json(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    print(f'  {out.stat().st_size/1024:.1f} KB written')


save_json(all_results, OUTPUT_JSON)
print(f'✓ Cell 9 complete — JSON → {OUTPUT_JSON}')

---
## Cell 10 — Save Excel
Three sheets: **Summary** · **MRZ Details** · **Full Text**

In [ ]:
def _style_header(ws) -> None:
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)


def _autofit(ws, max_w: int = 60) -> None:
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_w)


def build_excel(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    s_rows, m_rows, t_rows = [], [], []

    for r in results:
        md  = r.get('mrz_data') or {}
        val = md.get('validation', {}) if r.get('has_mrz') else {}
        ov  = val.get('overall_valid')

        s_rows.append({
            'file_name': r.get('file_name'), 'has_mrz': r.get('has_mrz'),
            'overall_mrz_valid': ov, 'elapsed_sec': r.get('elapsed_sec'),
            'error': r.get('error'),
        })
        t_rows.append({'file_name': r.get('file_name'), 'full_text': r.get('full_text')})

        if r.get('has_mrz') and md:
            pf = md.get('parsed_fields', {}) or {}
            m_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(md.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   ov,
                **{f'valid_{k}': v for k, v in val.items() if k != 'overall_valid'},
            })

    with pd.ExcelWriter(str(out), engine='openpyxl') as w:
        pd.DataFrame(s_rows).to_excel(w, sheet_name='Summary',     index=False)
        (pd.DataFrame(m_rows) if m_rows else pd.DataFrame(columns=['file_name'])
         ).to_excel(w, sheet_name='MRZ Details', index=False)
        pd.DataFrame(t_rows).to_excel(w, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(out))
    for ws in wb.worksheets:
        _style_header(ws)
        _autofit(ws)
    wb.save(str(out))
    print(f'  {out.stat().st_size/1024:.1f} KB written')


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Cell 10 complete — Excel → {OUTPUT_EXCEL}')

---
## Cell 11 — Final Summary

In [ ]:
df      = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
n       = len(df)
total_t = sum(r['elapsed_sec'] for r in all_results)

print('=' * 52)
print('  RESULTS SUMMARY')
print('=' * 52)
print(f'  Total documents    : {n}')
print(f'  With MRZ           : {int(df["has_mrz"].sum())}')
print(f'  MRZ valid          : {int(df["overall_mrz_valid"].sum())}')
print(f'  Errors             : {int(df["error"].notna().sum())}')
print(f'  Total time         : {total_t:.1f}s')
print(f'  Avg per document   : {total_t/max(n,1):.2f}s')
print('=' * 52)
print(f'  JSON  → {OUTPUT_JSON}')
print(f'  Excel → {OUTPUT_EXCEL}')
print('=' * 52)
print()
print('Download from: Domino → Jobs → Artifacts')
df